# IoT Device Classification & Location Prediction

## Multi-Output CNN using RF Spectrograms

This notebook trains a dual-head convolutional neural network that simultaneously predicts:
- **Device type** (6 classes): dooralarm, lora, microphone, mbus, sigfox, miwi
- **Device location** (3 classes): anotherroom, sameroom, upstairs

The model uses a shared CNN backbone with two separate softmax output heads.

## Setup and Instructions

### 1. Uploading the Data

To run this notebook, you need the RF recording data files (`.npy`) and the metadata manifest (`recordings_manifest.csv`).

#### A. Running on Google Colab
If you are running on Google Colab, you can upload the data in one of two ways:
1. **Direct Upload (Zip)**:
   * Zip your `data/` directory (containing `recordings_manifest.csv` and the `recordings` subfolders).
   * Upload `data.zip` to Colab's session storage.
   * Run the following command in a new cell to unzip it:
     ```python
     !unzip -q data.zip
     ```
2. **Google Drive Mount**:
   * Upload the `iot-device-classification` folder to your Google Drive.
   * Mount your drive in the notebook:
     ```python
     from google.colab import drive
     drive.mount('/content/drive')
     ```
   * Create a symlink to point the notebook to the data folder:
     ```python
     !ln -s "/content/drive/MyDrive/iot-device-classification/data" ./data
     ```

#### B. Running Locally
Ensure the data folder is placed at the root of the project:
```
iot-device-classification/
└── data/
    ├── recordings_manifest.csv
    └── recordings/
        ├── dooralarm/
        ├── lora/
        ├── microphone/
        ├── mbus/
        ├── sigfox/
        └── miwi/
```

### 2. How to Run the Code

1. **Install Dependencies**:
   Ensure you have the required libraries installed:
   ```bash
   pip install numpy scipy tensorflow matplotlib seaborn scikit-learn
   ```
2. **Execute Cells Sequentially**:
   * Run the **Configuration** cell to initialize hyperparameters, directories, and seeds.
   * Run the **Data Loading** cell to scan the manifest and verify the number of recordings for each split.
   * Run the **Signal Preprocessing** cell to define window segmentation and spectrogram conversion functions.
   * Run the **Build Multi-Output Dataset** cell to generate standardized training, validation, and test datasets.
   * Run the **Spectrogram Visualization** cell to view a sample spectrogram for each class.
   * Run the **Multi-Output CNN Architecture** cell to define the custom `StopGradientLayer` and compile the model.
   * Run the **Model Training** cell to fit the dual-head network using early stopping.
   * Run the **Test Evaluation** cell to produce classification reports.
   * Run the **Confusion Matrices** cell to plot the device and location confusion matrices.
   * Run the **Training Curves** cell to visualize training performance.
   * Run the **Single-File Inference** and **File-Level Test Evaluation** cells to verify ensemble predictions on whole recordings.


In [ ]:
import csv
import os
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from scipy import signal as sg
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)
from tensorflow.keras.layers import (
    BatchNormalization,
    Concatenate,
    Conv2D,
    Dense,
    Dropout,
    Flatten,
    GaussianNoise,
    GlobalAveragePooling2D,
    GlobalMaxPooling2D,
    Input,
    MaxPooling2D,
    ReLU,
)

# ============================================================
# Reproducibility
# ============================================================
np.random.seed(42)
tf.random.set_seed(42)

# ============================================================
# Configuration
# ============================================================
WINDOW_SIZE = 4096
NUM_CLASSES = 6
NUM_LOCATIONS = 3
BATCH_SIZE = 32
EPOCHS = 50
SEED = 42

CLASSES = [
    "dooralarm",
    "lora",
    "microphone",
    "mbus",
    "sigfox",
    "miwi",
]

LOCATIONS = [
    "anotherroom",
    "sameroom",
    "upstairs",
]

SPECTROGRAM_CONFIG = {
    "window": "hann",
    "nperseg": 256,
    "noverlap": 192,
    "nfft": 512,
    "scaling": "spectrum",
}

label_map = {idx: cls for idx, cls in enumerate(CLASSES)}
location_map = {idx: loc for idx, loc in enumerate(LOCATIONS)}

print("Configuration loaded.")
print(f"  Device classes: {CLASSES}")
print(f"  Location classes: {LOCATIONS}")

## Data Loading

We load recordings from a CSV manifest that specifies each file's device class, location scenario, capture ID, and train/validation/test split.

In [ ]:
@dataclass(frozen=True)
class Recording:
    """One independently captured signal file and its experimental metadata."""
    path: Path
    class_name: str
    scenario: str = "unspecified"
    capture_id: str = ""
    split: str | None = None


def load_manifest(manifest_path, classes=None, split=None):
    """Load recording assignments from a CSV manifest."""
    classes = classes or CLASSES
    required_columns = {"path", "class", "scenario", "capture_id", "split"}
    recordings = []

    with open(manifest_path, "r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        missing = required_columns.difference(reader.fieldnames or [])
        if missing:
            raise ValueError(f"Manifest is missing columns: {', '.join(sorted(missing))}")

        for row_number, row in enumerate(reader, start=2):
            class_name = row["class"].strip()
            if class_name not in classes:
                raise ValueError(f"Row {row_number} has unknown class: {class_name}")

            row_split = row["split"].strip()
            path = Path(row["path"].strip())
            if not path.exists():
                raise FileNotFoundError(f"Row {row_number} references missing file: {path}")

            recordings.append(
                Recording(
                    path=path,
                    class_name=class_name,
                    scenario=row["scenario"].strip() or "unspecified",
                    capture_id=row["capture_id"].strip(),
                    split=row_split or None,
                )
            )

    if split is not None:
        recordings = [r for r in recordings if r.split == split]

    if not recordings:
        requested = f" for split={split}" if split else ""
        raise ValueError(f"Manifest contains no recordings{requested}")

    return recordings


# Quick manifest summary
manifest_path = Path("data/recordings_manifest.csv")
all_recordings = load_manifest(manifest_path, classes=CLASSES)

split_counts = Counter(r.split or "unspecified" for r in all_recordings)
scenario_counts = Counter(r.scenario for r in all_recordings)
print(f"Total recordings: {len(all_recordings)}")
print(f"By split: {dict(sorted(split_counts.items()))}")
print(f"By scenario: {dict(sorted(scenario_counts.items()))}")

## Signal Preprocessing

Each `.npy` recording is segmented into non-overlapping windows, normalized, and converted to log-compressed spectrograms.

In [ ]:
def segment_signal(data, window_size=WINDOW_SIZE):
    """Split raw RF signal into fixed non-overlapping windows."""
    data = np.asarray(data).reshape(-1)
    num_samples = len(data) // window_size
    if num_samples == 0:
        return np.empty((0, window_size), dtype=data.dtype)
    return data[:num_samples * window_size].reshape(num_samples, window_size)


def normalize_windows(windows):
    """Normalize each RF window independently."""
    windows = np.asarray(windows)
    normalized = (windows - np.mean(windows, axis=1, keepdims=True)) / (
        np.std(windows, axis=1, keepdims=True) + 1e-8
    )
    if np.iscomplexobj(normalized):
        return normalized.astype(np.complex64)
    return normalized.astype(np.float32)


def create_spectrograms(windows):
    """Create normalized log-compressed spectrograms with a channel dimension."""
    specs = []
    for window in windows:
        _, _, spectrum = sg.spectrogram(window, **SPECTROGRAM_CONFIG)
        spec = np.log1p(spectrum)
        spec = (spec - np.mean(spec)) / (np.std(spec) + 1e-8)
        specs.append(spec.astype(np.float32))
    if not specs:
        return np.empty((0, 0, 0, 1), dtype=np.float32)
    return np.asarray(specs, dtype=np.float32)[..., np.newaxis]


def windows_to_model_input(windows):
    """Apply the full preprocessing path used by both training and inference."""
    if len(windows) == 0:
        return np.empty((0, 0, 0, 1), dtype=np.float32)
    normalized = normalize_windows(windows)
    return create_spectrograms(normalized)


print("Preprocessing functions defined.")

## Build Multi-Output Dataset

We convert recordings into `(X, y_device, y_location)` arrays, with optional per-class window cap and class balancing.

In [ ]:
def build_dataset_from_recordings(
    recordings,
    classes=CLASSES,
    locations=LOCATIONS,
    window_size=WINDOW_SIZE,
    balance=True,
    seed=SEED,
    max_windows_per_class=None,
):
    """Create window inputs from recordings.

    Returns (x, y_device, y_location) arrays.
    """
    entries_by_class = {cls: [] for cls in classes}
    windows_seen = {cls: 0 for cls in classes}
    rng = np.random.default_rng(seed)

    for recording in recordings:
        device_idx = classes.index(recording.class_name)
        scenario = recording.scenario
        location_idx = locations.index(scenario) if scenario in locations else -1

        raw = np.load(recording.path)
        windows = segment_signal(raw, window_size=window_size)
        if len(windows) == 0:
            raise ValueError(f"{recording.path} has fewer than {window_size} samples")

        if max_windows_per_class is None:
            for w in windows:
                entries_by_class[recording.class_name].append((w, device_idx, location_idx))
            continue

        samples = entries_by_class[recording.class_name]
        for window in windows:
            windows_seen[recording.class_name] += 1
            seen = windows_seen[recording.class_name]
            entry = (np.array(window, copy=True), device_idx, location_idx)
            if len(samples) < max_windows_per_class:
                samples.append(entry)
                continue
            candidate = int(rng.integers(0, seen))
            if candidate < max_windows_per_class:
                samples[candidate] = entry

    class_entries = {cls: entries for cls, entries in entries_by_class.items() if entries}
    balanced_limit = min(len(e) for e in class_entries.values()) if balance else None

    all_specs, all_dev_labels, all_loc_labels = [], [], []
    for cls in classes:
        if cls not in class_entries:
            continue
        entries = class_entries[cls]
        limit = balanced_limit if balance else len(entries)
        if max_windows_per_class is not None:
            limit = min(limit, max_windows_per_class)
        if len(entries) > limit:
            indices = np.sort(rng.choice(len(entries), size=limit, replace=False))
            entries = [entries[i] for i in indices]

        windows_array = np.asarray([e[0] for e in entries])
        specs = windows_to_model_input(windows_array)
        all_specs.append(specs)
        all_dev_labels.append(np.array([e[1] for e in entries], dtype=np.int64))
        all_loc_labels.append(np.array([e[2] for e in entries], dtype=np.int64))

    x = np.concatenate(all_specs, axis=0)
    y_device = np.concatenate(all_dev_labels, axis=0)
    y_location = np.concatenate(all_loc_labels, axis=0)

    return shuffle(x, y_device, y_location, random_state=seed)


MAX_WINDOWS_PER_CLASS = 2000

# Build train, validation, and test sets from manifest
print("Building training set...")
train_recs = load_manifest(manifest_path, classes=CLASSES, split="train")
X_train, y_dev_train, y_loc_train = build_dataset_from_recordings(
    train_recs, balance=True, max_windows_per_class=MAX_WINDOWS_PER_CLASS,
)

print("Building validation set...")
val_recs = load_manifest(manifest_path, classes=CLASSES, split="validation")
X_val, y_dev_val, y_loc_val = build_dataset_from_recordings(
    val_recs, balance=True, max_windows_per_class=MAX_WINDOWS_PER_CLASS,
)

print("Building test set...")
test_recs = load_manifest(manifest_path, classes=CLASSES, split="test")
X_test, y_dev_test, y_loc_test = build_dataset_from_recordings(
    test_recs, balance=True, max_windows_per_class=MAX_WINDOWS_PER_CLASS,
)

print(f"\nTrain: {X_train.shape}, device dist: {dict(zip(*np.unique(y_dev_train, return_counts=True)))}")
print(f"Val:   {X_val.shape}, device dist: {dict(zip(*np.unique(y_dev_val, return_counts=True)))}")
print(f"Test:  {X_test.shape}, device dist: {dict(zip(*np.unique(y_dev_test, return_counts=True)))}")
print(f"Train location dist: {dict(zip(*np.unique(y_loc_train, return_counts=True)))}")

## Spectrogram Visualization

Visual comparison of spectrograms from each device class.

In [ ]:
def plot_spectrogram(signal, title):
    """Plot a spectrogram of a raw signal window."""
    f, t, Sxx = sg.spectrogram(signal, nperseg=256, noverlap=128)
    Sxx_log = np.log1p(Sxx)

    plt.figure(figsize=(5, 4))
    plt.pcolormesh(t, f, Sxx_log, shading='gouraud')
    plt.title(title)
    plt.xlabel("Time")
    plt.ylabel("Frequency")
    plt.colorbar()
    plt.show()


# Plot one sample spectrogram per device class from training data
for idx, cls in enumerate(CLASSES):
    # Find a window belonging to this class
    mask = y_dev_train == idx
    if mask.any():
        sample_idx = np.where(mask)[0][0]
        # Reconstruct a raw window from a training recording
        rec = [r for r in train_recs if r.class_name == cls][0]
        raw = np.load(rec.path)
        windows = segment_signal(raw)
        if len(windows) > 0:
            plot_spectrogram(windows[0], f"{cls} ({rec.scenario})")

## Multi-Output CNN Architecture

The model uses a shared convolutional backbone with **avgmax pooling** (concatenated Global Average + Global Max Pooling) and two output heads:
- `device`: 6-class softmax for device identification
- `location`: 3-class softmax for location prediction

Loss weights: device=1.0, location=1.0

The location head has its own private Dense(128)→Dense(64) branch to learn environment-specific features independently from device features.

In [ ]:
@tf.keras.utils.register_keras_serializable(package="iot")
class StopGradientLayer(tf.keras.layers.Layer):
    """Pass-through layer that blocks gradient flow."""

    def call(self, inputs):
        return tf.stop_gradient(inputs)


def build_cnn(input_shape, num_classes, num_locations, learning_rate=3e-4, pooling="avgmax"):
    """Build and compile the multi-output spectrogram CNN."""
    inputs = Input(shape=input_shape)
    x = GaussianNoise(0.003)(inputs)

    # Block 1
    x = Conv2D(32, (3, 3), padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = MaxPooling2D((2, 2))(x)

    # Block 2
    x = Conv2D(64, (3, 3), padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = MaxPooling2D((2, 2))(x)

    # Block 3
    x = Conv2D(128, (3, 3), padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = MaxPooling2D((2, 2))(x)

    # Block 4
    x = Conv2D(256, (3, 3), padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    # Pooling
    if pooling == "flatten":
        x = Flatten()(x)
    elif pooling == "avg":
        x = GlobalAveragePooling2D()(x)
    elif pooling == "max":
        x = GlobalMaxPooling2D()(x)
    else:  # avgmax
        x = Concatenate()([GlobalAveragePooling2D()(x), GlobalMaxPooling2D()(x)])

    # Device head (shared dense layer)
    shared = Dense(256, activation="relu")(x)
    shared = Dropout(0.25)(shared)
    device_output = Dense(num_classes, activation="softmax", name="device")(shared)

    # Location head (detached from backbone via stop_gradient so
    # location gradients do NOT affect device-optimised backbone)
    loc_input = StopGradientLayer()(x)
    loc_branch = Dense(128, activation="relu")(loc_input)
    loc_branch = Dropout(0.3)(loc_branch)
    loc_branch = Dense(64, activation="relu")(loc_branch)
    location_output = Dense(num_locations, activation="softmax", name="location")(loc_branch)

    model = tf.keras.Model(inputs=inputs, outputs=[device_output, location_output])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss={
            "device": "categorical_crossentropy",
            "location": "categorical_crossentropy",
        },
        loss_weights={"device": 1.0, "location": 0.5},
        metrics={
            "device": ["accuracy"],
            "location": ["accuracy"],
        },
    )
    return model


model = build_cnn(
    input_shape=X_train.shape[1:],
    num_classes=NUM_CLASSES,
    num_locations=NUM_LOCATIONS,
    learning_rate=3e-4,
    pooling="avgmax",
)


## Model Training

Train with recording-level manifest splits, monitoring `val_device_accuracy` for early stopping and checkpointing.

In [ ]:
# One-hot encode labels
y_dev_train_cat = tf.keras.utils.to_categorical(y_dev_train, NUM_CLASSES)
y_dev_val_cat = tf.keras.utils.to_categorical(y_dev_val, NUM_CLASSES)
y_dev_test_cat = tf.keras.utils.to_categorical(y_dev_test, NUM_CLASSES)
y_loc_train_cat = tf.keras.utils.to_categorical(y_loc_train, NUM_LOCATIONS)
y_loc_val_cat = tf.keras.utils.to_categorical(y_loc_val, NUM_LOCATIONS)
y_loc_test_cat = tf.keras.utils.to_categorical(y_loc_test, NUM_LOCATIONS)

# Callbacks
lr_reducer = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1,
)

early_stop = EarlyStopping(
    monitor="val_device_accuracy",
    mode="max",
    patience=6,
    restore_best_weights=True,
    verbose=1,
)

checkpoint = ModelCheckpoint(
    "best_iot_classifier.keras",
    monitor="val_device_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1,
)

# Train
history = model.fit(
    X_train,
    {"device": y_dev_train_cat, "location": y_loc_train_cat},
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(
        X_val,
        {"device": y_dev_val_cat, "location": y_loc_val_cat},
    ),
    callbacks=[lr_reducer, early_stop, checkpoint],
    verbose=1,
)

## Test Evaluation

Evaluate both device and location prediction accuracy on the held-out test set.

In [ ]:
# Evaluate on test set
test_results = model.evaluate(
    X_test,
    {"device": y_dev_test_cat, "location": y_loc_test_cat},
    verbose=0,
)
test_metrics = dict(zip(model.metrics_names, test_results))

print("Test Metrics:")
for name, value in test_metrics.items():
    print(f"  {name}: {value:.4f}")

# Predictions
device_probs, location_probs = model.predict(X_test)
device_pred = np.argmax(device_probs, axis=1)
location_pred = np.argmax(location_probs, axis=1)

# Device Classification Report
print("\n" + "=" * 50)
print("DEVICE CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(
    y_dev_test, device_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASSES,
    zero_division=0,
))

# Location Classification Report
print("=" * 50)
print("LOCATION CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(
    y_loc_test, location_pred,
    labels=list(range(NUM_LOCATIONS)),
    target_names=LOCATIONS,
    zero_division=0,
))

## Confusion Matrices

Two separate confusion matrices: **Device** (6×6) and **Location** (3×3).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Device Confusion Matrix
cm_device = confusion_matrix(y_dev_test, device_pred, labels=list(range(NUM_CLASSES)))
sns.heatmap(
    cm_device,
    annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASSES, yticklabels=CLASSES,
    ax=axes[0],
)
axes[0].set_title("Device Confusion Matrix", fontsize=14)
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

# Location Confusion Matrix
cm_location = confusion_matrix(y_loc_test, location_pred, labels=list(range(NUM_LOCATIONS)))
sns.heatmap(
    cm_location,
    annot=True, fmt="d", cmap="Greens",
    xticklabels=LOCATIONS, yticklabels=LOCATIONS,
    ax=axes[1],
)
axes[1].set_title("Location Confusion Matrix", fontsize=14)
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.show()

## Training Curves

Device and location accuracy/loss over training epochs.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Device Accuracy
axes[0, 0].plot(history.history["device_accuracy"], label="Train")
axes[0, 0].plot(history.history["val_device_accuracy"], label="Validation")
axes[0, 0].set_title("Device Accuracy")
axes[0, 0].legend()

# Device Loss
axes[0, 1].plot(history.history["device_loss"], label="Train")
axes[0, 1].plot(history.history["val_device_loss"], label="Validation")
axes[0, 1].set_title("Device Loss")
axes[0, 1].legend()

# Location Accuracy
axes[1, 0].plot(history.history["location_accuracy"], label="Train")
axes[1, 0].plot(history.history["val_location_accuracy"], label="Validation")
axes[1, 0].set_title("Location Accuracy")
axes[1, 0].legend()

# Location Loss
axes[1, 1].plot(history.history["location_loss"], label="Train")
axes[1, 1].plot(history.history["val_location_loss"], label="Validation")
axes[1, 1].set_title("Location Loss")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## Single-File Inference

Run sliding-window ensemble prediction on a single `.npy` recording to predict both the device type and location.

In [ ]:
def sliding_windows(data, window_size=WINDOW_SIZE, step=1024):
    """Split a signal into overlapping windows for inference-time voting."""
    data = np.asarray(data).reshape(-1)
    if len(data) < window_size:
        return np.empty((0, window_size), dtype=data.dtype)
    windows = [
        data[start:start + window_size]
        for start in range(0, len(data) - window_size + 1, step)
    ]
    return np.asarray(windows)


def predict_signal(model, signal, classes=CLASSES, locations=LOCATIONS,
                   window_size=WINDOW_SIZE, step=1024):
    """Run multi-output inference on a raw signal."""
    windows = sliding_windows(signal, window_size=window_size, step=step)
    if len(windows) == 0:
        raise ValueError(f"Signal has {len(signal)} samples, fewer than window_size={window_size}")

    specs = windows_to_model_input(windows)
    raw_output = model.predict(specs)

    if isinstance(raw_output, list) and len(raw_output) == 2:
        raw_device_probs, raw_location_probs = raw_output
    else:
        raw_device_probs = raw_output
        raw_location_probs = None

    avg_device_probs = np.mean(raw_device_probs, axis=0)
    device_idx = int(np.argmax(avg_device_probs))

    result = {
        "device": classes[device_idx],
        "device_confidence": float(avg_device_probs[device_idx]),
        "windows": int(len(windows)),
    }

    if raw_location_probs is not None:
        avg_location_probs = np.mean(raw_location_probs, axis=0)
        location_idx = int(np.argmax(avg_location_probs))
        result["location"] = locations[location_idx]
        result["location_confidence"] = float(avg_location_probs[location_idx])

    return result


# Example: test on a recording from the test split
test_file = test_recs[0].path
print(f"Testing on: {test_file}")
print(f"True device: {test_recs[0].class_name}")
print(f"True location: {test_recs[0].scenario}")

test_signal = np.load(test_file)
result = predict_signal(model, test_signal)

print(f"\n--- Ensemble Results ({result['windows']} windows) ---")
print(f"Device Prediction:   {result['device']}")
print(f"Device Confidence:   {result['device_confidence'] * 100:.2f}%")
if "location" in result:
    print(f"Location Prediction: {result['location']}")
    print(f"Location Confidence: {result['location_confidence'] * 100:.2f}%")

## File-Level Test Evaluation

Evaluate each test recording independently using sliding-window ensemble prediction.

In [ ]:
print(f"Evaluating {len(test_recs)} test recordings...\n")

correct_dev, correct_loc = 0, 0
results_rows = []

for rec in test_recs:
    signal = np.load(rec.path)
    result = predict_signal(model, signal)

    dev_ok = int(result["device"] == rec.class_name)
    loc_ok = int(result.get("location", "") == rec.scenario)
    correct_dev += dev_ok
    correct_loc += loc_ok

    results_rows.append({
        "file": rec.path.name,
        "true_device": rec.class_name,
        "pred_device": result["device"],
        "device_conf": f"{result['device_confidence']:.2%}",
        "true_location": rec.scenario,
        "pred_location": result.get("location", "N/A"),
        "location_conf": f"{result.get('location_confidence', 0):.2%}",
        "dev_ok": "✓" if dev_ok else "✗",
        "loc_ok": "✓" if loc_ok else "✗",
    })

# Print results table
header = f"{'File':<40} {'True Dev':<12} {'Pred Dev':<12} {'D?':>3}  {'True Loc':<14} {'Pred Loc':<14} {'L?':>3}"
print(header)
print("-" * len(header))
for r in results_rows:
    print(f"{r['file']:<40} {r['true_device']:<12} {r['pred_device']:<12} {r['dev_ok']:>3}  {r['true_location']:<14} {r['pred_location']:<14} {r['loc_ok']:>3}")

dev_acc = correct_dev / len(test_recs)
loc_acc = correct_loc / len(test_recs)
print(f"\nFile-level Device Accuracy:   {dev_acc:.2%} ({correct_dev}/{len(test_recs)})")
print(f"File-level Location Accuracy: {loc_acc:.2%} ({correct_loc}/{len(test_recs)})")